In [ ]:
####SUBIR TABLA EVENTS DELTA LAKE######

In [ ]:
import os
import json
import time
import shutil
import requests
import pandas as pd
from datetime import datetime, UTC
from deltalake.writer import write_deltalake

BASE_URL = "https://gamma-api.polymarket.com"
LOCAL_DELTA_PATH = "./polymarket/bronze/events"


# ---------- RESET: elimina tabla delta previa ----------
def reset_delta_folder(path: str):
    if os.path.exists(path):
        print(f"🧨 Eliminando Delta previo en: {path}")
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)


def fetch_events_page(
    limit: int,
    offset: int,
    # parámetros típicos del endpoint /events (opcional)
    active: bool | None = None,
    closed: bool | None = None,
    order: str | None = None,
    ascending: bool | None = None,
    retries: int = 3,
    backoff_s: float = 1.5,
    timeout_s: int = 180,
):
    url = f"{BASE_URL}/events"

    params = {"limit": limit, "offset": offset}  # paginación soportada [web:16]
    if active is not None:
        params["active"] = str(active).lower()
    if closed is not None:
        params["closed"] = str(closed).lower()
    if order is not None:
        params["order"] = order
    if ascending is not None:
        params["ascending"] = str(ascending).lower()

    for attempt in range(1, retries + 1):
        try:
            print(f"GET {url} params={params} (attempt {attempt}/{retries})")
            r = requests.get(url, params=params, timeout=timeout_s)
            r.raise_for_status()

            data = r.json()
            # algunos endpoints pueden envolver en {"data":[...]} (lo dejamos robusto)
            if isinstance(data, dict) and "data" in data:
                data = data["data"]

            if not isinstance(data, list):
                raise ValueError(f"Formato inesperado: {type(data)}")

            return data

        except Exception as e:
            if attempt == retries:
                raise
            sleep = backoff_s * attempt
            print(f"⚠️ fallo: {e} -> reintento en {sleep:.1f}s")
            time.sleep(sleep)


def build_canonical_columns(sample_pages=10, limit=100, **fetch_kwargs):
    cols = set()
    offset = 0
    for _ in range(sample_pages):
        rows = fetch_events_page(limit=limit, offset=offset, **fetch_kwargs)
        if not rows:
            break
        df = pd.json_normalize(rows)
        cols.update(df.columns.tolist())
        offset += limit

    cols.add("_ingestion_ts")
    canonical = sorted(cols)
    print(f"📌 Schema canónico construido con {len(canonical)} columnas")
    return canonical


def sanitize_objects_to_json(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == "object":
            sample = df[col].dropna().head(50).tolist()
            if any(isinstance(x, (dict, list)) for x in sample):
                df[col] = df[col].apply(
                    lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
                )
    return df


def align_to_schema(df: pd.DataFrame, canonical_cols: list) -> pd.DataFrame:
    df = df.copy()

    for c in canonical_cols:
        if c not in df.columns:
            df[c] = None

    extra = [c for c in df.columns if c not in canonical_cols]
    if extra:
        df = df.drop(columns=extra)

    return df[canonical_cols]


def force_no_null_only_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    null_only_cols = [c for c in df.columns if df[c].isna().all()]
    if null_only_cols:
        print(f"🛠️ Fix Null-only cols (cast a string): {null_only_cols}")
        for c in null_only_cols:
            df[c] = df[c].astype("string")
    df = df.where(pd.notnull(df), None)
    return df


def ingest_events_to_delta(
    limit=100,
    max_pages=5000,
    sleep_s=0.0,
    sample_pages_for_schema=10,
    # filtros opcionales
    active: bool | None = None,
    closed: bool | None = None,
    order: str | None = "id",
    ascending: bool | None = False,
):
    reset_delta_folder(LOCAL_DELTA_PATH)

    canonical_cols = build_canonical_columns(
        sample_pages=sample_pages_for_schema,
        limit=limit,
        active=active,
        closed=closed,
        order=order,
        ascending=ascending,
    )

    offset = 0
    total = 0
    first_write = True

    for page in range(max_pages):
        rows = fetch_events_page(
            limit=limit,
            offset=offset,
            active=active,
            closed=closed,
            order=order,
            ascending=ascending,
        )
        if not rows:
            print(f"[events] fin en page={page}, offset={offset}")
            break

        df = pd.json_normalize(rows)
        df["_ingestion_ts"] = datetime.now(UTC)

        df = sanitize_objects_to_json(df)
        df = align_to_schema(df, canonical_cols)
        df = force_no_null_only_columns(df)

        mode = "overwrite" if first_write else "append"
        print(f"[events] escribiendo page={page} filas={len(df)} mode={mode}")
        write_deltalake(LOCAL_DELTA_PATH, df, mode=mode)

        first_write = False
        total += len(df)
        offset += limit

        print(f"[events] total acumulado: {total}")

        if sleep_s:
            time.sleep(sleep_s)

    print("✅ Delta Lake EVENTS generado en local.")
    print(f"📁 Sube esta carpeta a S3: {LOCAL_DELTA_PATH}")
    print(f"📌 Total filas events: {total}")


if __name__ == "__main__":
    # Ejemplo recomendado para “eventos activos” (ajústalo a tu caso) [web:22]
    ingest_events_to_delta(
        limit=100,
        max_pages=5000,
        sleep_s=0.0,
        sample_pages_for_schema=10,
        active=True,
        closed=False,
        order="id",
        ascending=False,
    )


In [ ]:
#####SUBIR TABLA MARKETS Y TAGS DELTA LAKE#######

In [ ]:
import os
import json
import time
import shutil
import requests
import pandas as pd
from datetime import datetime, UTC
from deltalake.writer import write_deltalake

BASE_URL = "https://gamma-api.polymarket.com"
LOCAL_DELTA_PATH = "./polymarket/bronze/tags"


# ---------- RESET: elimina tabla delta previa ----------
def reset_delta_folder(path: str):
    if os.path.exists(path):
        print(f"🧨 Eliminando Delta previo en: {path}")
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)


def fetch_tags(retries: int = 3, backoff_s: float = 1.5, timeout_s: int = 180):
    """
    GET /tags
    Devuelve lista de tags. [web:2]
    """
    url = f"{BASE_URL}/tags"

    for attempt in range(1, retries + 1):
        try:
            print(f"GET {url} (attempt {attempt}/{retries})")
            r = requests.get(url, timeout=timeout_s)
            r.raise_for_status()

            data = r.json()
            # Por consistencia con otros endpoints, si viniera envuelto en {"data": [...]}
            if isinstance(data, dict) and "data" in data:
                data = data["data"]

            if not isinstance(data, list):
                raise ValueError(f"Formato inesperado: {type(data)}")

            return data

        except Exception as e:
            if attempt == retries:
                raise
            sleep = backoff_s * attempt
            print(f"⚠️ fallo: {e} -> reintento en {sleep:.1f}s")
            time.sleep(sleep)


def sanitize_objects_to_json(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == "object":
            sample = df[col].dropna().head(50).tolist()
            if any(isinstance(x, (dict, list)) for x in sample):
                df[col] = df[col].apply(
                    lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
                )
    return df


def align_to_schema(df: pd.DataFrame, canonical_cols: list) -> pd.DataFrame:
    df = df.copy()

    for c in canonical_cols:
        if c not in df.columns:
            df[c] = None

    extra = [c for c in df.columns if c not in canonical_cols]
    if extra:
        df = df.drop(columns=extra)

    return df[canonical_cols]


def force_no_null_only_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    null_only_cols = [c for c in df.columns if df[c].isna().all()]
    if null_only_cols:
        print(f"🛠️ Fix Null-only cols (cast a string): {null_only_cols}")
        for c in null_only_cols:
            df[c] = df[c].astype("string")
    df = df.where(pd.notnull(df), None)
    return df


def build_canonical_columns_from_tags(rows: list):
    """
    Construye schema canónico a partir de la respuesta completa de /tags. [web:2]
    """
    df = pd.json_normalize(rows)
    cols = set(df.columns.tolist())
    cols.add("_ingestion_ts")
    canonical = sorted(cols)
    print(f"📌 Schema canónico construido con {len(canonical)} columnas")
    return canonical


def ingest_tags_to_delta():
    reset_delta_folder(LOCAL_DELTA_PATH)

    rows = fetch_tags()
    if not rows:
        print("⚠️ /tags devolvió 0 filas; no se genera Delta.")
        return

    canonical_cols = build_canonical_columns_from_tags(rows)

    df = pd.json_normalize(rows)
    df["_ingestion_ts"] = datetime.now(UTC)

    df = sanitize_objects_to_json(df)
    df = align_to_schema(df, canonical_cols)
    df = force_no_null_only_columns(df)

    print(f"[tags] escribiendo filas={len(df)} mode=overwrite")
    write_deltalake(LOCAL_DELTA_PATH, df, mode="overwrite")

    print("✅ Delta Lake TAGS generado en local.")
    print(f"📁 Sube esta carpeta a S3: {LOCAL_DELTA_PATH}")
    print(f"📌 Total filas tags: {len(df)}")


if __name__ == "__main__":
    ingest_tags_to_delta()


In [ ]:
#SUBIR TABLA MARKETS#########

In [ ]:
import os
import json
import time
import shutil
import requests
import pandas as pd
from datetime import datetime, UTC
from deltalake.writer import write_deltalake

BASE_URL = "https://gamma-api.polymarket.com"
LOCAL_DELTA_PATH = "./polymarket/bronze/markets"


# ---------- RESET: elimina tabla delta previa ----------
def reset_delta_folder(path: str):
    if os.path.exists(path):
        print(f"🧨 Eliminando Delta previo en: {path}")
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)


def fetch_markets_page(limit: int, offset: int, retries: int = 3, backoff_s: float = 1.5):
    url = f"{BASE_URL}/markets"
    params = {"limit": limit, "offset": offset}

    for attempt in range(1, retries + 1):
        try:
            print(f"GET {url} params={params} (attempt {attempt}/{retries})")
            r = requests.get(url, params=params, timeout=180)
            r.raise_for_status()

            data = r.json()
            if isinstance(data, dict) and "data" in data:
                data = data["data"]

            if not isinstance(data, list):
                raise ValueError(f"Formato inesperado: {type(data)}")

            return data

        except Exception as e:
            if attempt == retries:
                raise
            sleep = backoff_s * attempt
            print(f"⚠️ fallo: {e} -> reintento en {sleep:.1f}s")
            time.sleep(sleep)


def build_canonical_columns(sample_pages=10, limit=500):
    cols = set()
    offset = 0
    for _ in range(sample_pages):
        rows = fetch_markets_page(limit=limit, offset=offset)
        if not rows:
            break
        df = pd.json_normalize(rows)
        cols.update(df.columns.tolist())
        offset += limit

    cols.add("_ingestion_ts")
    canonical = sorted(cols)
    print(f"📌 Schema canónico construido con {len(canonical)} columnas")
    return canonical


def sanitize_objects_to_json(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == "object":
            sample = df[col].dropna().head(50).tolist()
            if any(isinstance(x, (dict, list)) for x in sample):
                df[col] = df[col].apply(
                    lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
                )
    return df


def align_to_schema(df: pd.DataFrame, canonical_cols: list) -> pd.DataFrame:
    df = df.copy()

    for c in canonical_cols:
        if c not in df.columns:
            df[c] = None

    extra = [c for c in df.columns if c not in canonical_cols]
    if extra:
        df = df.drop(columns=extra)

    return df[canonical_cols]


def force_no_null_only_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    null_only_cols = [c for c in df.columns if df[c].isna().all()]
    if null_only_cols:
        print(f"🛠️ Fix Null-only cols (cast a string): {null_only_cols}")
        for c in null_only_cols:
            df[c] = df[c].astype("string")
    df = df.where(pd.notnull(df), None)
    return df


def ingest_markets_to_delta(limit=500, max_pages=2000, sleep_s=0.0, sample_pages_for_schema=10):
    # ✅ clave: limpiar tabla anterior
    reset_delta_folder(LOCAL_DELTA_PATH)

    canonical_cols = build_canonical_columns(sample_pages=sample_pages_for_schema, limit=limit)

    offset = 0
    total = 0
    first_write = True

    for page in range(max_pages):
        rows = fetch_markets_page(limit=limit, offset=offset)
        if not rows:
            print(f"[markets] fin en page={page}, offset={offset}")
            break

        df = pd.json_normalize(rows)
        df["_ingestion_ts"] = datetime.now(UTC)

        df = sanitize_objects_to_json(df)
        df = align_to_schema(df, canonical_cols)
        df = force_no_null_only_columns(df)

        mode = "overwrite" if first_write else "append"
        print(f"[markets] escribiendo page={page} filas={len(df)} mode={mode}")

        write_deltalake(LOCAL_DELTA_PATH, df, mode=mode)

        first_write = False
        total += len(df)
        offset += limit

        print(f"[markets] total acumulado: {total}")

        if sleep_s:
            time.sleep(sleep_s)

    print("✅ Delta Lake MARKETS generado en local.")
    print(f"📁 Sube esta carpeta a S3: {LOCAL_DELTA_PATH}")
    print(f"📌 Total filas markets: {total}")


if __name__ == "__main__":
    ingest_markets_to_delta(limit=500, max_pages=2000, sleep_s=0.0, sample_pages_for_schema=10)

In [ ]:
# SUBIR ARCHIVOS MARKET, EVENTS, TAGS, ETC


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
import boto3
from botocore.exceptions import ClientError

#clavess de acceso
AWS_ACCESS_KEY_ID = os.getenv('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = os.getenv('AWS_SECRET_ACCESS_KEY')
AWS_REGION = os.getenv('AWS_REGION')
BUCKET     = "lasalle-bigdata-2025-2026"
TAG        = "edgard_fernandez"
LOCAL_BASE = "./polymarket/bronze"
S3_PREFIX  = f"{TAG}/polymarket/bronze"

TABLAS = ["markets", "events", "series", "tags"]


def upload_delta_table(s3_client, local_dir: str, s3_prefix: str):
    local_dir = os.path.abspath(local_dir)
    if not os.path.exists(local_dir):
        print(f"⚠️  Carpeta no encontrada, se omite: {local_dir}")
        return 0

    uploaded = 0
    for root, _, files in os.walk(local_dir):
        for fname in files:
            local_path = os.path.join(root, fname)
            rel_path   = os.path.relpath(local_path, local_dir).replace(os.sep, "/")
            s3_key     = f"{s3_prefix.rstrip('/')}/{rel_path}"

            try:
                s3_client.upload_file(local_path, BUCKET, s3_key)
                print(f"  ✅ {s3_key}")
                uploaded += 1
            except ClientError as e:
                print(f"  ❌ Error subiendo {s3_key}: {e}")

    return uploaded


def main():
    session = boto3.session.Session(
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
        region_name=AWS_REGION,
    )
    s3 = session.client("s3")

    print(f"🚀 Subiendo Delta tables a s3://{BUCKET}/{S3_PREFIX}/\n")
    total = 0

    for tabla in TABLAS:
        local_dir = os.path.join(LOCAL_BASE, tabla)
        prefix    = f"{S3_PREFIX}/{tabla}"
        print(f"📦 [{tabla}] → s3://{BUCKET}/{prefix}")
        n = upload_delta_table(s3, local_dir, prefix)
        print(f"   → {n} ficheros subidos\n")
        total += n

    print(f"✅ Subida completada. Total ficheros: {total}")
    print(f"📍 Ruta S3: s3://{BUCKET}/{S3_PREFIX}/")


if __name__ == "__main__":
    main()


In [ ]:
#REPORTE DE VOLUMETRIA

In [ ]:
#IMPORTAR TODAS LAS TABLAS

In [ ]:
import pandas as pd
import json
from deltalake import DeltaTable

# Cargar las 4 tablas Delta
df_tags    = DeltaTable("./polymarket/bronze/tags").to_pandas()
df_series  = DeltaTable("./polymarket/bronze/series").to_pandas()
df_events  = DeltaTable("./polymarket/bronze/events").to_pandas()
df_markets = DeltaTable("./polymarket/bronze/markets").to_pandas()

print("✅ Tablas cargadas")


In [ ]:
#CONTAR TODOS LOS REGISTROS DE LAS TABLAS

In [ ]:
print("=" * 50)
print("📊 1. CANTIDAD DE REGISTROS POR ENTIDAD")
print("=" * 50)

resumen = pd.DataFrame({
    "Entidad":   ["tags", "series", "events", "markets"],
    "Registros": [len(df_tags), len(df_series), len(df_events), len(df_markets)],
    "Columnas":  [df_tags.shape[1], df_series.shape[1], df_events.shape[1], df_markets.shape[1]],
})
print(resumen.to_string(index=False))


In [ ]:
print("\n" + "=" * 50)
print("📊 2. DISTRIBUCIÓN DE MERCADOS (activos vs cerrados)")
print("=" * 50)

# Markets
mkt_active = df_markets["active"].value_counts(dropna=False).rename("markets")
print("\n--- MARKETS ---")
print(mkt_active)
print(f"  Activos   : {df_markets['active'].sum():,}")
print(f"  Cerrados  : {(~df_markets['active'].fillna(False)).sum():,}")

# Events
if "active" in df_events.columns and "closed" in df_events.columns:
    print("\n--- EVENTS ---")
    ev_dist = df_events.groupby(
        [df_events["active"].fillna(False), df_events["closed"].fillna(False)]
    ).size().reset_index()
    ev_dist.columns = ["active", "closed", "count"]
    print(ev_dist.to_string(index=False))


In [ ]:
#MEDIAS Y COMPARACIONES ENTRE TABLAS

In [ ]:
print("\n" + "=" * 50)
print("📊 3. ANÁLISIS DE RELACIONES")
print("=" * 50)

# --- Mercados por Evento ---
if "eventId" in df_markets.columns:
    mkts_per_event = (
        df_markets.groupby("eventId")
        .size()
        .reset_index(name="num_markets")
    )
    print("\n📌 Mercados por Evento:")
    print(f"  Eventos con markets: {len(mkts_per_event):,}")
    print(f"  Media markets/evento: {mkts_per_event['num_markets'].mean():.2f}")
    print(f"  Máx  markets/evento: {mkts_per_event['num_markets'].max()}")
    print(f"  Mín  markets/evento: {mkts_per_event['num_markets'].min()}")
    print("\n  Top 10 eventos con más mercados:")
    print(
        mkts_per_event.nlargest(10, "num_markets")
        .merge(df_events[["id", "title"]], left_on="eventId", right_on="id", how="left")
        [["title", "num_markets"]]
        .to_string(index=False)
    )

# --- Events por Serie ---
if "seriesSlug" in df_events.columns:
    events_per_series = (
        df_events[df_events["seriesSlug"].notna()]
        .groupby("seriesSlug")
        .size()
        .reset_index(name="num_events")
    )
    print("\n📌 Eventos por Serie (seriesSlug):")
    print(f"  Series con eventos: {len(events_per_series):,}")
    print(f"  Media eventos/serie: {events_per_series['num_events'].mean():.2f}")
    print(f"  Máx eventos/serie : {events_per_series['num_events'].max()}")
    print("\n  Top 10 series con más eventos:")
    print(events_per_series.nlargest(10, "num_events").to_string(index=False))


In [ ]:
#GENERAR EL REPORTE DE VOLUMETRIA

In [ ]:
!pip install fpdf


In [ ]:
from fpdf import FPDF

# ============================================================
# 5) GENERAR REPORTE PDF DE VOLUMETRÍA GLOBAL
# ============================================================

class PDF(FPDF):
    def header(self):
        self.set_font("Arial", "B", 16)
        self.cell(0, 10, "Reporte de Volumetria - Ecosistema Polymarket", ln=True, align="C")
        self.ln(5)

    def chapter_title(self, title):
        self.set_font("Arial", "B", 12)
        self.set_text_color(0, 51, 102) # Azul oscuro
        self.cell(0, 10, title, ln=True)
        self.ln(2)

    def chapter_body(self, text):
        self.set_font("Arial", "", 11)
        self.set_text_color(0, 0, 0)
        self.multi_cell(0, 6, text)
        self.ln(5)

# Instanciar el PDF
pdf = PDF()
pdf.add_page()

# --- 1. CANTIDAD DE REGISTROS POR ENTIDAD ---
pdf.chapter_title("1. Cantidad de Registros por Entidad (Capa Bronze)")
vol_text = (
    f"- Tags: {len(df_tags):,} registros ({df_tags.shape[1]} columnas)\n"
    f"- Series: {len(df_series):,} registros ({df_series.shape[1]} columnas)\n"
    f"- Eventos: {len(df_events):,} registros ({df_events.shape[1]} columnas)\n"
    f"- Mercados: {len(df_markets):,} registros ({df_markets.shape[1]} columnas)"
)
pdf.chapter_body(vol_text)

# --- 2. DISTRIBUCIÓN DE MERCADOS Y EVENTOS ---
pdf.chapter_title("2. Distribucion (Activos vs Cerrados)")

# Cálculo de mercados
activos_mkt = int(df_markets['active'].fillna(False).sum()) if 'active' in df_markets.columns else 0
cerrados_mkt = len(df_markets) - activos_mkt

dist_text = (
    f"--- MERCADOS ---\n"
    f"- Activos: {activos_mkt:,}\n"
    f"- Cerrados/Inactivos: {cerrados_mkt:,}\n"
    f"- Total: {len(df_markets):,}\n"
)

# Cálculo de eventos
if "active" in df_events.columns:
    activos_ev = int(df_events['active'].fillna(False).sum())
    cerrados_ev = len(df_events) - activos_ev
    dist_text += (
        f"\n--- EVENTOS ---\n"
        f"- Activos: {activos_ev:,}\n"
        f"- Cerrados/Inactivos: {cerrados_ev:,}\n"
        f"- Total: {len(df_events):,}"
    )

pdf.chapter_body(dist_text)

# --- 3. ANÁLISIS DE RELACIONES ---
pdf.chapter_title("3. Analisis de Relaciones")
rel_text = ""

if len(df_events) > 0:
    ratio_mk_ev = len(df_markets) / len(df_events)
    rel_text += f"- Promedio global: {ratio_mk_ev:.2f} mercados por cada evento.\n"

if len(df_series) > 0:
    ratio_ev_ser = len(df_events) / len(df_series)
    rel_text += f"- Promedio global: {ratio_ev_ser:.2f} eventos por cada serie.\n"

# Revisar relaciones explícitas en el campo 'events' dentro de markets
if "events" in df_markets.columns:
    con_evento = int(df_markets["events"].notna().sum())
    rel_text += f"\n- Hay {con_evento:,} mercados que contienen referencias explicitas a eventos."

if not rel_text:
    rel_text = "No se encontraron datos suficientes para establecer relaciones."

pdf.chapter_body(rel_text)

# Guardar el PDF
pdf_filename = "Reporte_Volumetria.pdf"
pdf.output(pdf_filename)

print(f"✅ Reporte de volumetría exportado exitosamente como: {pdf_filename}")


In [ ]:
#FASE 3 LIMPIEZA DE DATOS

In [ ]:
#RECOGER DATOS DE IRAN VS EEUU

import re
import numpy as np
import pandas as pd
from deltalake import DeltaTable

#LEER DELTALAKE PARA BUSCAR EN LA INFORMACION DE POLYMARKET
df_tags   = DeltaTable("./polymarket/bronze/tags").to_pandas()
df_series = DeltaTable("./polymarket/bronze/series").to_pandas()
df_events = DeltaTable("./polymarket/bronze/events").to_pandas()

dtm = DeltaTable("./polymarket/bronze/markets")
available_cols = [f.name for f in dtm.schema().fields]  # compatible con tu deltalake

wanted_cols = [
    "id","slug","question","title","description",
    "active","closed",
    "volumeNum","liquidityNum",
    "events","outcomes","outcomePrices",
    "category","subcategory","marketType","marketGroup"
]
markets_cols = [c for c in wanted_cols if c in available_cols]
df_markets = dtm.to_pandas(columns=markets_cols)

print("✅ Tablas cargadas (local)")
print("tags  :", df_tags.shape)
print("series:", df_series.shape)
print("events:", df_events.shape)
print("markets:", df_markets.shape, "| cols:", len(df_markets.columns))

#FILTRAR POR PAISES Y LA PALABRA VS, TODO LO RELACIONADO CON IRAN VS EEUU
terms = [
    "eeuu-vs-iran", "eeuu vs iran", "us vs iran", "u.s. vs iran", "us iran",
    "us strikes iran", "estados unidos", "united states", "iran", "irán"
]
rx = re.compile("|".join(re.escape(t) for t in terms), re.IGNORECASE)

def filter_df(df, cols):
    cols = [c for c in cols if c in df.columns]
    if not cols:
        return df.iloc[0:0].copy()
    mask = False
    for c in cols:
        mask = mask | df[c].astype("string").str.contains(rx, na=False)
    return df.loc[mask].copy()

tags_f    = filter_df(df_tags,   ["label","slug","name"])
series_f  = filter_df(df_series, ["title","slug","description"])
events_f  = filter_df(df_events, ["title","slug","description"])
markets_f = filter_df(df_markets,["question","title","slug","description"])

print("\n✅ Filtrado por tema")
print("tags_f  :", tags_f.shape)
print("series_f:", series_f.shape)
print("events_f:", events_f.shape)
print("markets_f:", markets_f.shape)

#ELIMINAR NULOS Y NONE DE LA INFORMACIÓN EXTRAIDA
NULL_LIKE = {"": np.nan, "None": np.nan, "none": np.nan, "null": np.nan, "NULL": np.nan, "nan": np.nan}

def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 3.1) Normalizar nulos "de texto"
    df.replace(NULL_LIKE, inplace=True)

    # 3.2) Limpiar strings (solo object/string)
    txt_cols = df.select_dtypes(include=["object", "string"]).columns
    for c in txt_cols:
        df[c] = (
            df[c].astype("string")
                 .str.replace(r"\s+", " ", regex=True)
                 .str.strip()
        )
        df[c] = df[c].replace({"": np.nan})  # por si quedó vacío tras strip

    # 3.3) Fechas típicas (si existen)
    date_cols = [c for c in df.columns if any(k in c.lower() for k in ["date", "time", "created", "updated", "start", "end"])]
    for c in date_cols:
        # si no es parseable, queda NaT (equivale a nulo de fecha)
        df[c] = pd.to_datetime(df[c], errors="coerce", utc=True)

    # 3.4) Booleans típicos
    for c in ["active", "closed", "archived", "approved"]:
        if c in df.columns:
            df[c] = df[c].fillna(False).astype(bool)

    # 3.5) Duplicados
    if "id" in df.columns:
        df = df.drop_duplicates(subset=["id"])
    else:
        df = df.drop_duplicates()

    return df

tags_clean    = clean_df(tags_f)
series_clean  = clean_df(series_f)
events_clean  = clean_df(events_f)
markets_clean = clean_df(markets_f)

print("\n✅ Limpieza aplicada")
print("tags_clean  :", tags_clean.shape)
print("series_clean:", series_clean.shape)
print("events_clean:", events_clean.shape)
print("markets_clean:", markets_clean.shape)

#EXPORTAR EN LAS TABLAS (MAS ADELANTE SE AÑADIRAN VALORES A LA TABLA TAG PORQUE TIENE MUY POCA INFORMACIÓN)
tags_clean.to_csv("tags_eeuu_iran_clean.csv", index=False, encoding="utf-8")
series_clean.to_csv("series_eeuu_iran_clean.csv", index=False, encoding="utf-8")
events_clean.to_csv("events_eeuu_iran_clean.csv", index=False, encoding="utf-8")
markets_clean.to_csv("markets_eeuu_iran_clean.csv", index=False, encoding="utf-8")

print("\n✅ Exportados CSV limpios:")
print(" - tags_eeuu_iran_clean.csv")
print(" - series_eeuu_iran_clean.csv")
print(" - events_eeuu_iran_clean.csv")
print(" - markets_eeuu_iran_clean.csv")


In [ ]:
#NEONDB SUBIDA DE ARCHIVOS

In [ ]:
import os, json
from dotenv import load_dotenv
load_dotenv()
import pandas as pd
from sqlalchemy import create_engine, text

# ✅ Connection string (NO lo publiques). Mejor:
# NEON_CONN = os.environ["NEON_CONN"]
NEON_CONN = os.getenv('NEONCONN')
engine = create_engine(NEON_CONN)

# ✅ Tus ficheros (están en el mismo directorio del notebook)
CSV_MARKETS = "markets_eeuu_iran_clean.csv"
CSV_EVENTS  = "events_eeuu_iran_clean.csv"
CSV_SERIES  = "series_eeuu_iran_clean.csv"
CSV_TAGS    = "tags_eeuu_iran_clean.csv"

# -----------------------------
# Helpers
# -----------------------------
def to_num(s):
    return pd.to_numeric(s, errors="coerce")

def to_ts(s):
    return pd.to_datetime(s, errors="coerce", utc=True)

def to_bool(s):
    def conv(x):
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return None
        if isinstance(x, bool):
            return x
        t = str(x).strip().lower()
        if t in ("true","1","yes","y","t"):  return True
        if t in ("false","0","no","n","f"):  return False
        return None
    return s.map(conv)

def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def parse_json_cell(cell):
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return None
    if isinstance(cell, (list, dict)):
        return cell
    if isinstance(cell, str):
        s = cell.strip()
        if (s.startswith("{") and s.endswith("}")) or (s.startswith("[") and s.endswith("]")):
            try:
                return json.loads(s)
            except:
                return None
    return None

def extract_first_id_from_events(cell):
    obj = parse_json_cell(cell)
    if isinstance(obj, list) and obj:
        x = obj[0]
        if isinstance(x, dict) and "id" in x:
            try: return int(x["id"])
            except: return None
        if isinstance(x, (int, float)) and pd.notna(x):
            try: return int(x)
            except: return None
    if isinstance(obj, dict) and "id" in obj:
        try: return int(obj["id"])
        except: return None
    return None

#ASIGNACION DE TIPOS PARA LAS TABLAS
markets = pd.read_csv(CSV_MARKETS, low_memory=False)
events  = pd.read_csv(CSV_EVENTS,  low_memory=False)
series  = pd.read_csv(CSV_SERIES,  low_memory=False)
tags    = pd.read_csv(CSV_TAGS,    low_memory=False)

print("📦 Leído CSV:",
      "markets", markets.shape,
      "events", events.shape,
      "series", series.shape,
      "tags", tags.shape)

# AÑADIR DIMENSIONES CORRECTAS A LAS TABLAS
ev_end_col   = first_existing(events, ["endDate","end_date","end_ts","endsAt","endTime","closeTime"])
ev_cat_col   = first_existing(events, ["category","subcategory","series","sport","type"])
ev_title_col = first_existing(events, ["title","name","question"])

dim_event = pd.DataFrame({
    "event_id": to_num(events["id"]) if "id" in events.columns else None,
    "title": events[ev_title_col] if ev_title_col else None,
    "category": events[ev_cat_col] if ev_cat_col else None,
    "end_ts": to_ts(events[ev_end_col]) if ev_end_col else None,
}).dropna(subset=["event_id"]).drop_duplicates(subset=["event_id"])
dim_event["event_id"] = dim_event["event_id"].astype("int64")

# ----- DIM TAG -----
tg_parent_col = first_existing(tags, ["parentId","parent_id","parentTagId","parent_tag_id"])
tg_name_col   = first_existing(tags, ["name","label","title"])
tg_slug_col   = first_existing(tags, ["slug","tag","code"])

dim_tag = pd.DataFrame({
    "tag_id": to_num(tags["id"]) if "id" in tags.columns else None,
    "name": tags[tg_name_col] if tg_name_col else None,
    "slug": tags[tg_slug_col] if tg_slug_col else None,
    "parent_tag_id": to_num(tags[tg_parent_col]) if tg_parent_col else None
}).dropna(subset=["tag_id"]).drop_duplicates(subset=["tag_id"])

if len(dim_tag):
    dim_tag["tag_id"] = dim_tag["tag_id"].astype("int64")
    if "parent_tag_id" in dim_tag.columns:
        dim_tag["parent_tag_id"] = dim_tag["parent_tag_id"].astype("Int64")

# ----- DIM MARKET -----
mk_question_col = first_existing(markets, ["question","title","name"])
mk_end_col      = first_existing(markets, ["endDate","end_date","end_ts","endTime","closeTime"])
mk_vol_col      = first_existing(markets, ["volumeNum","volume","volumeUSD"])
mk_liq_col      = first_existing(markets, ["liquidityNum","liquidity","liquidityUSD"])
mk_cat_col      = first_existing(markets, ["category","subcategory","marketType","marketGroup"])

# event_id: si ya existe, úsalo; si no, intenta extraer de "events"
if "event_id" in markets.columns:
    mk_event_id = to_num(markets["event_id"])
else:
    evs_col = first_existing(markets, ["events","event","eventId","event_id"])
    if evs_col in ("events","event") and evs_col in markets.columns:
        mk_event_id = to_num(markets[evs_col].apply(extract_first_id_from_events))
    else:
        mk_event_id = to_num(markets[evs_col]) if evs_col and evs_col in markets.columns else None

dim_market = pd.DataFrame({
    "market_id": to_num(markets["id"]) if "id" in markets.columns else None,
    "question": markets[mk_question_col] if mk_question_col else None,
    "active": to_bool(markets["active"]).fillna(False).astype(bool) if "active" in markets.columns else None,
    "closed": to_bool(markets["closed"]).fillna(False).astype(bool) if "closed" in markets.columns else None,
    "archived": to_bool(markets["archived"]).fillna(False).astype(bool) if "archived" in markets.columns else None,
    "category": markets[mk_cat_col] if mk_cat_col else None,
    "liquidity": to_num(markets[mk_liq_col]) if mk_liq_col else None,
    "volume": to_num(markets[mk_vol_col]) if mk_vol_col else None,
    "event_id": mk_event_id,
    "end_ts": to_ts(markets[mk_end_col]) if mk_end_col else None,
}).dropna(subset=["market_id"]).drop_duplicates(subset=["market_id"])

if len(dim_market):
    dim_market["market_id"] = dim_market["market_id"].astype("int64")
    dim_market["event_id"] = dim_market["event_id"].astype("Int64")

# ----- DIM TIME (por día) -----
dim_time = dim_market[["end_ts"]].dropna().copy()
if len(dim_time):
    dim_time["date"] = pd.to_datetime(dim_time["end_ts"], utc=True).dt.date
    dim_time = dim_time.dropna().drop_duplicates()
    dim_time["time_id"] = pd.to_datetime(dim_time["date"]).dt.strftime("%Y%m%d").astype(int)
    dim_time["year"] = pd.to_datetime(dim_time["date"]).dt.year
    dim_time["month"] = pd.to_datetime(dim_time["date"]).dt.month
    dim_time["day"] = pd.to_datetime(dim_time["date"]).dt.day
    dim_time = dim_time[["time_id","date","year","month","day"]].drop_duplicates()

print("🧼 GOLD shapes:",
      "dim_time", dim_time.shape,
      "dim_event", dim_event.shape,
      "dim_tag", dim_tag.shape,
      "dim_market", dim_market.shape)

# CREAR TABLAS DENTRO DE NEONDB CON SQL
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS polymarket;"))
    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS polymarket.dim_time(
      time_id INT PRIMARY KEY,
      date DATE,
      year INT,
      month INT,
      day INT
    );
    """))
    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS polymarket.dim_event(
      event_id BIGINT PRIMARY KEY,
      title TEXT,
      category TEXT,
      end_ts TIMESTAMPTZ
    );
    """))
    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS polymarket.dim_tag(
      tag_id BIGINT PRIMARY KEY,
      name TEXT,
      slug TEXT,
      parent_tag_id BIGINT
    );
    """))
    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS polymarket.dim_market(
      market_id BIGINT PRIMARY KEY,
      question TEXT,
      active BOOLEAN,
      closed BOOLEAN,
      archived BOOLEAN,
      category TEXT,
      liquidity NUMERIC,
      volume NUMERIC,
      event_id BIGINT,
      end_ts TIMESTAMPTZ
    );
    """))

# CARGAS LAS TABLAS EN NEON DB UNA VEEZ CREADAS ANTERIORMENTE
dim_time.to_sql("dim_time", engine, schema="polymarket", if_exists="replace", index=False, chunksize=5000, method="multi")
dim_event.to_sql("dim_event", engine, schema="polymarket", if_exists="replace", index=False, chunksize=5000, method="multi")
dim_tag.to_sql("dim_tag", engine, schema="polymarket", if_exists="replace", index=False, chunksize=5000, method="multi")
dim_market.to_sql("dim_market", engine, schema="polymarket", if_exists="replace", index=False, chunksize=5000, method="multi")

print("✅ NeonDB cargado (GOLD): polymarket.dim_time, dim_event, dim_tag, dim_market")
